# 01 - Load and clean the KBA FZ 10.1 files

**Question for the whole project:** how is the German new-car market moving to electric cars, and which brands drive it?

This notebook shows *why* the cleaning code in `src/kba_fz10.py` looks the way it does. The KBA Excel files are made for people to read, not for analysis, so they need care before any number can be trusted.

In [2]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, "../src")
from kba_fz10 import parse_file, load_folder

pd.set_option("display.width", 200, "display.max_columns", 30)
RAW = Path("../data/raw")
sorted(p.name for p in RAW.glob("*.xlsx"))[:5], len(list(RAW.glob("*.xlsx")))

(['fz10_2024_09.xlsx',
  'fz10_2024_10.xlsx',
  'fz10_2024_11.xlsx',
  'fz10_2024_12.xlsx',
  'fz10_2025_01.xlsx'],
 24)

## 1. What the raw sheet looks like

Sheet `FZ 10.1`, read without any header handling. Note the title rows on top, the two-level header, and that the brand is only written on the first row of each block.

In [4]:
raw = pd.read_excel(RAW / "fz10_2026_08.xlsx", sheet_name="FZ 10.1", header=None, dtype=object)
print(raw.shape)
raw.iloc[5:20, 1:9]

(552, 45)


,1,2,3,4,5,6,7,8
5,FZ 10.1 Neuzulassungen von Personenkraftwagen ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Marke,Modellreihe,Insgesamt,NaN,NaN,mit Dieselantrieb,NaN,NaN
8,NaN,NaN,August \n2026,Jan. - August 2026,Anteil \nin %,August \n2026,Jan. - August 2026,Anteil \nin %
9,ALFA ROMEO,GIULIA,33,244,7.7,1,23,8.6
10,NaN,JUNIOR,192,1894,59.6,-,-,-
11,NaN,STELVIO,67,577,18.2,19,193,72.6
12,NaN,TONALE,47,464,14.6,4,50,18.8
13,ALFA ROMEO ZUSAMMEN,NaN,339,3179,0.2,24,266,0.1
14,ALPINE,A110,3,96,9.3,-,-,-


## 2. Five things that would silently break the numbers

| Trap | What the parser does |
|---|---|
| Rows like `AUDI ZUSAMMEN` are subtotals | Dropped from the data; used only to check that the model rows add up |
| Brand is written only on the first row of its block | Forward-filled |
| Zero is written as `-`, counts can be text | Converted to numbers |
| Each fuel group has 3 columns (month, year to date, share %) | Only the month column is used, otherwise years would be double counted |
| Fuel groups **overlap** (hybrid incl. plug-in contains plug-in hybrid) | Only non-overlapping groups are used; petrol is derived |
| Older files put the group names one row above `Marke` | Header search looks in the rows above and on the `Marke` row |
| `SONSTIGE` (other brands) sometimes has no model rows | Kept as one row, otherwise about 1,000 cars a month would vanish |

The last row was found by a check, not by luck: see section 4.

In [6]:
tidy, report = parse_file(RAW / "fz10_2026_08.xlsx")
print(report)
tidy.head(10)

ParseReport(file='fz10_2026_08.xlsx', month='2026-08', model_rows=471, reconciliation_gaps=0, negative_residual_rows=0, grand_total_parsed=212563, grand_total_official=212563)


,month,brand,model,fuel_key,fuel_category,registrations
0,2026-08-01,ALFA ROMEO,GIULIA,bev,Electric (BEV),0
1,2026-08-01,ALFA ROMEO,JUNIOR,bev,Electric (BEV),74
2,2026-08-01,ALFA ROMEO,STELVIO,bev,Electric (BEV),0
3,2026-08-01,ALFA ROMEO,TONALE,bev,Electric (BEV),0
4,2026-08-01,ALPINE,A110,bev,Electric (BEV),0
5,2026-08-01,ALPINE,A290,bev,Electric (BEV),69
6,2026-08-01,ALPINE,A390,bev,Electric (BEV),24
7,2026-08-01,ALPINE,SONSTIGE,bev,Electric (BEV),0
8,2026-08-01,ASTON MARTIN,SONSTIGE,bev,Electric (BEV),0
9,2026-08-01,AUDI,A1,bev,Electric (BEV),0


## 3. Proof that the fuel groups overlap

For the Audi total row: hybrid *including* plug-in should equal hybrid *without* plug-in plus plug-in hybrid. If it does, adding all columns together would double count.

In [8]:
# Audi ZUSAMMEN row: month columns of "Hybrid incl. plug-in" (col 9), "Hybrid ohne plug-in" (col 18), "Plug-in" (col 27)
audi = raw[raw[1].astype(str).str.upper() == "AUDI ZUSAMMEN"].iloc[0]
incl, without, plug = audi[9], audi[18], audi[27]
print(f"hybrid incl. plug-in = {incl}   |   hybrid w/o plug-in + plug-in = {without} + {plug} = {without + plug}")

hybrid incl. plug-in = 6521   |   hybrid w/o plug-in + plug-in = 3182 + 3339 = 6521


So the five categories used in this project do not overlap:

`BEV` + `Plug-in hybrid` + `Hybrid (no plug-in)` + `Diesel` + `Petrol & other` = total registrations.

`Petrol & other` is calculated as total minus the other four, because the file has no petrol column. It also holds small volumes of gas, hydrogen and other drives.

In [10]:
audi_tidy = tidy[tidy["brand"] == "AUDI"].groupby("fuel_category")["registrations"].sum()
print(audi_tidy)
print("Audi total:", audi_tidy.sum())

fuel_category
Diesel                  592
Electric (BEV)         4319
Hybrid (no plug-in)    3182
Petrol & other         1373
Plug-in hybrid         3339
Name: registrations, dtype: int64
Audi total: 12805


## 4. The check that found a bug

KBA prints its own grand total (`NEUZULASSUNGEN INSGESAMT`) at the bottom of each file. The first version of the parser was **1,082 cars short in August 2026** because it treated the "other brands" line as a subtotal and dropped it. Comparing against KBA's own total caught it. The parser now stores this comparison for every file.

In [12]:
data, reports, skipped = load_folder(RAW)
check = pd.DataFrame([r.__dict__ for r in reports])[["month", "grand_total_parsed", "grand_total_official", "reconciliation_gaps"]]
check["matches_KBA"] = check["grand_total_parsed"] == check["grand_total_official"]
print("skipped files:", skipped)
check

skipped files: []


,month,grand_total_parsed,grand_total_official,reconciliation_gaps,matches_KBA
0,2024-09,208848,208848,0,True
1,2024-10,231992,231992,0,True
2,2024-11,244544,244544,0,True
3,2024-12,224721,224721,0,True
4,2025-01,207640,207640,0,True
5,2025-02,203434,203434,0,True
6,2025-03,253497,253497,0,True
7,2025-04,242728,242728,0,True
8,2025-05,239297,239297,0,True
9,2025-06,256193,256193,0,True


In [13]:
assert check["matches_KBA"].all(), "A month does not match KBA's official total"
print("All", len(check), "months match KBA's official total.")

All 24 months match KBA's official total.


## 5. Build the analysis dataset

`python src/build_dataset.py` does the same, merges the renamed brand (SsangYong became KGM in April 2025), and writes the CSV and SQLite database used in the next notebook.

In [15]:
import subprocess
print(subprocess.run([sys.executable, "../src/build_dataset.py"], capture_output=True, text=True, cwd="../src").stdout)

OK  24 months (2024-09 to 2026-08), 51,975 rows, 5,732,720 registrations
    every month matches KBA's official 'NEUZULASSUNGEN INSGESAMT'
    wrote /Users/aishwaryasonawane/Documents/german-ev-market-analysis/data/processed/registrations_tidy.csv, registrations.sqlite, parse_report.csv

